In [1]:
import numpy as np
import torch as torch
import os
import copy
import random
from scipy.optimize import nnls,curve_fit,leastsq
import SimpleITK as sitk
import nibabel as nib
import time
import statsmodels.api as sm

In [2]:

def WLS_FIT(inputs,b_list,NSA):
    generate_parameters=np.zeros((inputs.shape[0],1))
    for i in range(inputs.shape[0]):
        if inputs[i][0]<18.42068:
            res_ols = sm.OLS(inputs[i],b_list).fit()
            res_wls = sm.WLS(inputs[i],b_list,weights=NSA*(res_ols.fittedvalues**2)).fit()
            generate_parameters[i]=res_wls.params
        else:
            generate_parameters[i]=0
    
    return generate_parameters

def bootstrap_rmse(residuals, n_boot=1000):
    mean=np.sqrt(np.mean(residuals**2))
    boot_rmses = []
    for _ in range(n_boot):
        sample = np.random.choice(residuals, size=len(residuals), replace=True)
        boot_rmses.append(np.sqrt(np.mean(sample**2)))
    lower = np.percentile(boot_rmses, 2.5)
    upper = np.percentile(boot_rmses, 97.5)
    return mean,lower, upper

In [3]:
data_dict=torch.load(r"G:\zhouxinxiang\MRL_HN\simulation\ADC_data_dict.pth")
parameter_dict=torch.load(r"G:\zhouxinxiang\MRL_HN\simulation\ADC_parameter_dict.pth")
error_dict=torch.load(r"G:\zhouxinxiang\MRL_HN\simulation\ADC_error_dict.pth")

In [19]:
val_signals,val_parameters=data_dict['val']['Multi']
val_signals=val_signals[:,1:]/val_signals[:,0].reshape(-1,1)
val_signals=-np.log(val_signals+1e-8)

parameter=WLS_FIT(np.array(val_signals[:,[3,4,5,6]]),torch.tensor([150,300,400,500]).numpy(),np.array([4,6,6,8]))

print(bootstrap_rmse((parameter-np.array(val_parameters))[:,0]))

(0.00011327675675537502, 0.00011177420647487054, 0.00011502704265268724)


In [20]:
val_signals,val_parameters=data_dict['val']['Average']
s1=np.array(val_signals[:,1])
s2=np.array(val_signals[:,2])

parameter=-(np.log(s2/s1)/(500-150))
print(bootstrap_rmse((parameter-np.array(val_parameters))[:,0]))

(0.0004731949313047615, 0.00047063113820124114, 0.00047600929830644774)
